# Semi-Autonomous Mode

Unlike supervised mode - where checkpoints are placed at *phase boundaries* defined in advance - semi-autonomous mode makes a dynamic risk decision on *every individual tool call*. Low-risk actions execute immediately without any human involvement. High-risk actions pause and escalate to a human reviewer before executing.

The result: humans are consulted only when their judgment genuinely adds value, not on a fixed schedule and not on every action.

```
Agent receives task
    │
    ▼
For each planned tool call:
    │
    ├── Classify action risk
    │       │
    │       ├── LOW RISK  ──→  Execute immediately (no human involved)
    │       │
    │       └── HIGH RISK ──→  Pause, escalate to human
    │                               │
    │                               ├── Approved ──→  Execute
    │                               └── Rejected ──→  Replan
    │
    ▼
Continue with next tool call
```

**Autonomy level: 3 - Risk-classified autonomy**
Semi-autonomous mode is the right choice when a task mixes clearly safe operations-— reads, analyses, drafts - with risky ones - sends, deletes, publishes - and full-phase checkpoints would interrupt the safe majority unnecessarily.

In [1]:
import os
import json as _json
import re
from enum import Enum
from dataclasses import dataclass
from typing import Callable

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

### Initialize the language model

In [2]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, api_key=os.getenv('OPENAI_API_KEY'))
print('LLM ready:', llm.model_name)

LLM ready: gpt-4o-mini


## The risk classification model
Before any tool call executes, the agent needs to answer a binary question: is this action safe enough to run automatically, or does it require a human to review it first? The answer lives in two small data structures.

`RiskLevel` is a four-value scale from low through critical. In practice most deployments only use two tiers - low and high - because the escalation decision itself is binary. The intermediate values exist for logging and reporting: a high-risk action that was approved many times may be worth reclassifying to medium for future sessions.

`RiskClassification` carries the full verdict: the level, a one-sentence reason, and the `requires_approval` flag that drives the routing decision. Keeping the reason in the dataclass means the human reviewer always sees *why* an action was escalated, not just that it was — which is essential for calibrating the classifier over time.

In [3]:
class RiskLevel(Enum):
    LOW      = 'low'
    MEDIUM   = 'medium'
    HIGH     = 'high'
    CRITICAL = 'critical'


@dataclass
class RiskClassification:
    """The result of classifying a proposed tool call.

    Attributes:
        level: Severity of the risk.
        reason: One-sentence explanation of the classification.
        requires_approval: Whether a human must approve before execution.
    """
    level: RiskLevel
    reason: str
    requires_approval: bool


# Illustrate the model with two contrasting examples
low_example = RiskClassification(
    level=RiskLevel.LOW,
    reason="Tool 'search_knowledge_base' is a read-only operation with no side effects.",
    requires_approval=False,
)
high_example = RiskClassification(
    level=RiskLevel.HIGH,
    reason="Tool 'send_email' sends a message to external recipients — irreversible.",
    requires_approval=True,
)

for ex in [low_example, high_example]:
    print(f'Level: {ex.level.value:<10} | Approval required: {ex.requires_approval} | {ex.reason}')

Level: low        | Approval required: False | Tool 'search_knowledge_base' is a read-only operation with no side effects.
Level: high       | Approval required: True | Tool 'send_email' sends a message to external recipients — irreversible.


The two fields that drive behaviour are `level` and `requires_approval`. The classifier sets both; the agent loop reads only `requires_approval` to decide whether to escalate. `level` is preserved in the action log for calibration work: if the log shows that all `high` escalations are being approved without modification, the threshold may be too conservative, and some tools might warrant reclassification to `low`.

## Approach 1: rule-based risk classifier
A rule-based classifier uses explicit allow and deny lists to classify tool calls deterministically. Classification is an O(1) dict lookup with no LLM call required, making it fast and auditable: the risk rules are visible in code and any change has an immediate, predictable effect.

This approach is appropriate when the risk of a tool is independent of its arguments - `send_email` is always high-risk regardless of what the email says, and `search_knowledge_base` is always low-risk regardless of the query. For tools whose risk *depends on their arguments*, the LLM-driven approach in the next cell is a better fit.

In [4]:
class RuleBasedRiskClassifier:
    """Classify action risk using explicit tool allow/deny lists.

    Any tool in HIGH_RISK_TOOLS requires approval; any tool in LOW_RISK_TOOLS
    executes automatically. Unknown tools default to requiring approval — a
    safe fallback for tools not yet classified.
    """

    # Tools that modify external state, communicate with people, or are irreversible
    HIGH_RISK_TOOLS = {
        'send_email', 'send_sms', 'post_to_slack',
        'delete_record', 'delete_file', 'drop_table',
        'deploy_to_production', 'publish_to_web',
        'charge_payment', 'transfer_funds',
        'write_to_crm',
    }

    # Tools that only observe — reads, searches, status checks
    LOW_RISK_TOOLS = {
        'search_web', 'read_file', 'query_database',
        'search_knowledge_base', 'read_sales_data',
        'list_directory', 'get_config', 'check_status',
        'generate_report',
    }

    def classify(self, tool_name: str, tool_args: dict) -> RiskClassification:
        """Classify the risk of a proposed tool call.

        Args:
            tool_name: Name of the tool the agent wants to call.
            tool_args: Arguments the agent prepared for the call.

        Returns:
            A RiskClassification with level, reason, and approval requirement.
        """
        if tool_name in self.HIGH_RISK_TOOLS:
            return RiskClassification(
                level=RiskLevel.HIGH,
                reason=f"'{tool_name}' is in the high-risk list (irreversible or external action).",
                requires_approval=True,
            )
        if tool_name in self.LOW_RISK_TOOLS:
            return RiskClassification(
                level=RiskLevel.LOW,
                reason=f"'{tool_name}' is a read-only or safe internal operation.",
                requires_approval=False,
            )
        # Unknown tool — default to requiring approval (fail-safe behaviour)
        return RiskClassification(
            level=RiskLevel.MEDIUM,
            reason=f"'{tool_name}' is not in any known list — defaulting to approval required.",
            requires_approval=True,
        )


# Demonstrate against a representative set of tools
rule_classifier = RuleBasedRiskClassifier()

test_tools = [
    ('search_knowledge_base', {'query': 'Q4 sales summary'}),
    ('read_sales_data',        {'period': 'q4-2025'}),
    ('generate_report',        {'title': 'Q4 Summary', 'data': '...'}),
    ('send_email',             {'to': 'vp@company.com', 'subject': 'Q4 Report', 'body': '...'}),
    ('write_to_crm',           {'customer_id': 'C-001', 'updates': {'tier': 'enterprise'}}),
    ('mystery_tool',           {'arg': 'value'}),
]

print(f"{'Tool':<30} {'Risk':<10} {'Approval Required'}")
print('─' * 55)
for name, args in test_tools:
    r = rule_classifier.classify(name, args)
    gate = '🔒 yes' if r.requires_approval else '🔓 no'
    print(f'{name:<30} {r.level.value:<10} {gate}')

Tool                           Risk       Approval Required
───────────────────────────────────────────────────────
search_knowledge_base          low        🔓 no
read_sales_data                low        🔓 no
generate_report                low        🔓 no
send_email                     high       🔒 yes
write_to_crm                   high       🔒 yes
mystery_tool                   medium     🔒 yes


The classifier correctly separates the read-only tools (no approval needed) from the external-write tools (approval required). The `mystery_tool` falls through to the medium-risk default - requiring approval rather than silently executing. This fail-safe default is intentional: an unknown tool should not auto-execute until it has been explicitly classified.

## Approach 2: LLM-driven risk classifier
A rule-based classifier is fast and predictable, but it classifies by tool *name* alone. An LLM-driven classifier can reason about the *arguments*: the same `execute_query` tool carries very different risk profiles depending on whether the SQL is a five-row SELECT or a DELETE affecting 48,000 rows. Rule-based classification cannot distinguish these; LLM-driven classification can.

The tradeoff is one additional LLM call per tool invocation, adding roughly one second of latency and a small cost. This is worthwhile when the argument-level context substantially changes the risk, and can be mitigated with caching for repeated tool/argument patterns.

In [5]:
class LLMRiskClassifier:
    """Classify action risk using the LLM's contextual reasoning over arguments.

    Best suited to tools where the risk depends on what the tool is called WITH,
    not just which tool is being called. Falls back to high-risk on parse failure.
    """

    # The prompt asks the LLM to evaluate five risk dimensions and respond in JSON
    RISK_PROMPT = """You are a risk assessment system for a semi-autonomous AI agent.

Evaluate the following proposed tool call and classify its risk.

Tool: {tool_name}
Arguments: {tool_args}

Risk dimensions to evaluate:
1. Reversibility — can this action be easily undone?
2. Scope — how many records, users, or systems are affected?
3. Sensitivity — does this involve PII, financial data, or regulated content?
4. Visibility — will external parties see the result?
5. Side effects — does it trigger communications or transactions?

Respond with JSON only (no markdown or extra text):
{{"risk_level": "low or high", "requires_approval": true or false, "reason": "one sentence"}}"""

    def __init__(self, llm):
        self.llm = llm

    def classify(self, tool_name: str, tool_args: dict) -> RiskClassification:
        """Classify risk by prompting the LLM for a contextual assessment.

        Args:
            tool_name: The name of the tool to be called.
            tool_args: The arguments that will be passed to the tool.

        Returns:
            A RiskClassification parsed from the LLM's JSON response.
        """
        prompt = self.RISK_PROMPT.format(
            tool_name=tool_name,
            tool_args=_json.dumps(tool_args, indent=2),
        )
        response = self.llm.invoke(prompt)

        # Strip any markdown fences the model might accidentally include
        content = re.sub(r'```json?\s*|\s*```', '', response.content).strip()

        try:
            data = _json.loads(content)
            return RiskClassification(
                level=RiskLevel[data['risk_level'].upper()],
                reason=data['reason'],
                requires_approval=data['requires_approval'],
            )
        except (KeyError, _json.JSONDecodeError, ValueError):
            # Parse failure: default to high risk — never silently auto-execute on error
            return RiskClassification(
                level=RiskLevel.HIGH,
                reason='Could not parse risk assessment — defaulting to approval required.',
                requires_approval=True,
            )


# The LLM classifier's strength is distinguishing risk based on argument content.
# The same tool 'execute_query' has very different risk profiles depending on the SQL:
llm_classifier = LLMRiskClassifier(llm)

ambiguous_cases = [
    ('execute_query', {'sql': 'SELECT id, name FROM users LIMIT 10'}),
    ('execute_query', {'sql': 'DELETE FROM users WHERE last_login < \'2020-01-01\'', 'estimated_rows': 48000}),
]

for tool_name, args in ambiguous_cases:
    risk = llm_classifier.classify(tool_name, args)
    print(f'Tool : {tool_name}')
    print(f'Args : {list(args.values())[0][:70]}')
    print(f'Risk : {risk.level.value} | Approval: {risk.requires_approval}')
    print(f'Why  : {risk.reason}')
    print()

Tool : execute_query
Args : SELECT id, name FROM users LIMIT 10
Risk : low | Approval: False
Why  : The query retrieves a limited number of non-sensitive user identifiers and names without modifying any data.

Tool : execute_query
Args : DELETE FROM users WHERE last_login < '2020-01-01'
Risk : high | Approval: True
Why  : The action permanently deletes a significant number of user records, which is irreversible and may involve sensitive personal information.



The same tool name produces two completely different risk classifications based on argument content: the SELECT query with a small limit is classified as low-risk and would auto-execute; the DELETE query affecting tens of thousands of rows is classified as high-risk and would escalate for approval. This context-sensitivity is what the rule-based classifier cannot provide for argument-dependent risk.

In practice many deployments use a hybrid: a fast rule-based check first, with an LLM override only for tools in a designated "context-sensitive" list. This keeps the majority of classifications at O(1) cost while applying LLM reasoning where it genuinely adds value.

## Mock tools: a mixed-risk catalog
The tool set below intentionally mixes low-risk reads with high-risk writes, mirroring the structure of a real reporting workflow. Data retrieval, analysis, and report generation are safe to automate. Sending external communications or updating persistent records requires human oversight. This asymmetry is exactly the scenario semi-autonomous mode is designed for.

In [6]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search the internal knowledge base. Read-only, no side effects.

    Args:
        query: Search terms.

    Returns:
        Matching content snippet.
    """
    docs = {
        'q4 sales':        'Q4 2025: 2,340 new customers acquired, up 18% YoY. Enterprise segment led growth.',
        'email template':  'Weekly report template: Subject = "Weekly Metrics | {date}", Body = "{summary}".',
        'sales team lead': 'VP of Sales contact: vp-sales@company.com. Prefers concise bullet-point summaries.',
    }
    for key, value in docs.items():
        if key in query.lower():
            return value
    return 'No relevant results found for that query.'


@tool
def read_sales_data(period: str) -> str:
    """Fetch sales data for a reporting period. Read-only.

    Args:
        period: Reporting period identifier (e.g. 'q4-2025').

    Returns:
        Sales data summary string.
    """
    data = {
        'q4-2025': '1,200 enterprise, 840 mid-market, 300 SMB customers. Total ARR added: $4.2M.',
        'q3-2025': '980 enterprise, 720 mid-market, 250 SMB customers. Total ARR added: $3.6M.',
    }
    return data.get(period, f'No data found for period: {period}')


@tool
def generate_report(title: str, content: str) -> str:
    """Format data into a structured report. Internal only — no external side effects.

    Args:
        title: Report title.
        content: Report body content.

    Returns:
        A formatted report string.
    """
    return f'REPORT: {title}\nGenerated: 2026-03-27\n{"-" * 40}\n{content}\n{"-" * 40}'


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to an external recipient. Irreversible external action.

    Args:
        to: Recipient email address.
        subject: Email subject line.
        body: Email body text.

    Returns:
        Delivery confirmation.
    """
    return f'[MOCK] Email delivered → {to} | Subject: "{subject}" | {len(body)} chars'


@tool
def write_to_crm(customer_id: str, updates: dict) -> str:
    """Update a customer record in the CRM. Modifies persistent data.

    Args:
        customer_id: Customer identifier to update.
        updates: Dictionary of field updates to apply.

    Returns:
        Update confirmation.
    """
    return f'[MOCK] CRM updated → customer {customer_id}: {updates}'


ALL_TOOLS = [search_knowledge_base, read_sales_data, generate_report, send_email, write_to_crm]
TOOL_MAP  = {t.name: t for t in ALL_TOOLS}

# Show each tool's risk classification so the auto-execute vs escalate split is clear
print('Tool catalog with risk classification:')
for t in ALL_TOOLS:
    r = rule_classifier.classify(t.name, {})
    gate = '🔒 escalate' if r.requires_approval else '🔓 auto'
    print(f'  {gate}  {t.name:<30} ({r.level.value} risk)')

Tool catalog with risk classification:
  🔓 auto  search_knowledge_base          (low risk)
  🔓 auto  read_sales_data                (low risk)
  🔓 auto  generate_report                (low risk)
  🔒 escalate  send_email                     (high risk)
  🔒 escalate  write_to_crm                   (high risk)


The catalog splits cleanly: the three read and compute tools auto-execute; the two external-write tools require escalation. This ratio - roughly 60% auto-execute, 40% escalate - is typical of a well-tuned semi-autonomous deployment. Escalation rates above 70% suggest thresholds are too conservative; rates below 10% may mean the classifier is missing genuine risks.

## Escalation callbacks
When the risk classifier flags a tool call as high-risk, the agent pauses and invokes an escalation callback. The callback presents the proposed action to a human and returns an `(approved, feedback)` tuple - identical in structure to supervised mode's approval callbacks, so the same infrastructure can serve both.

The callback is injected into the agent rather than hardcoded, making the same agent portable across environments. A format helper function is shared between callbacks to ensure the escalation display is consistent regardless of which callback is active.

In [7]:
# Type alias: any function with this signature can serve as an escalation callback
EscalationCallback = Callable[[str, dict, RiskClassification], tuple[bool, str]]


def _format_escalation(tool_name: str, tool_args: dict, risk: RiskClassification) -> str:
    """Format a human-readable escalation request for any callback to display."""
    return (
        f'\nESCALATION REQUIRED'
        f'\n{"-" * 45}'
        f'\nProposed action : {tool_name}'
        f'\nParameters      : {_json.dumps(tool_args, indent=2)}'
        f'\nRisk level      : {risk.level.value.upper()}'
        f'\nReason          : {risk.reason}'
        f'\n{"-" * 45}'
    )


def auto_escalation(
    tool_name: str, tool_args: dict, risk: RiskClassification
) -> tuple[bool, str]:
    """Auto-approves all escalations. Use for testing and demonstrations.

    Args:
        tool_name: The tool awaiting approval.
        tool_args: Arguments for the tool call.
        risk: The risk classification result.

    Returns:
        (True, '') — always approved with no feedback.
    """
    print(_format_escalation(tool_name, tool_args, risk))
    print('>>> AUTO-APPROVED (demo mode) <<<')
    return True, ''


def console_escalation(
    tool_name: str, tool_args: dict, risk: RiskClassification
) -> tuple[bool, str]:
    """Interactive escalation via stdin. Use in interactive notebook sessions.

    Args:
        tool_name: The tool awaiting approval.
        tool_args: Arguments for the tool call.
        risk: The risk classification result.

    Returns:
        (approved, feedback) based on human input.
    """
    print(_format_escalation(tool_name, tool_args, risk))
    while True:
        choice = input('Approve? (y / n / f for feedback): ').strip().lower()
        if choice == 'y':
            return True, ''
        elif choice == 'n':
            reason = input('Reason for rejection: ')
            return False, reason
        elif choice.startswith('f'):
            feedback = input('Feedback (agent will replan incorporating this): ')
            return True, feedback
        print('Please enter y, n, or f.')


print('Escalation callbacks ready: auto_escalation, console_escalation')

Escalation callbacks ready: auto_escalation, console_escalation


Both callbacks receive the same three arguments - tool name, tool arguments, and the risk classification - and return the same tuple. The `_format_escalation` helper ensures the escalation display is consistent: whoever is reviewing always sees the proposed action, its exact parameters, and the reason it was flagged. This is especially important for building reviewer trust; a bare approval prompt with no context leads to rubber-stamping rather than genuine review.

## The semi-autonomous agent
The core of semi-autonomous mode is a risk gate inserted into the standard agentic loop. The loop itself is identical to all other modes: call the LLM, execute tool calls, feed results back, repeat until no more tool calls. The difference is a single classification check before each execution - the risk gate.

Three outcomes are possible for every tool call: auto-execute if low-risk; escalate and execute if high-risk and approved; escalate and skip if high-risk and rejected (feeding a rejection message back to the LLM so it can replan). The `action_log` records every decision with its classification, outcome, and reason - the raw material for calibrating risk thresholds over time.

In [8]:
SEMI_AUTO_SYSTEM = """You are an intelligent agent operating in Semi-Autonomous Mode.

You have full access to all available tools. Your tool calls are evaluated for risk
before execution — this happens transparently in the infrastructure layer. You will
receive the tool result either way (auto-executed for low-risk; escalated-and-executed
or rejected for high-risk).

When a tool call is rejected, use the feedback to replan and propose an alternative
approach that achieves the same goal with lower risk.

Always produce a clear completion summary when the task is done.
"""


class SemiAutonomousAgent:
    """Agent with per-action risk classification built into the agentic loop.

    Low-risk tool calls execute immediately. High-risk calls escalate to a human
    before execution. Rejections are fed back to the LLM for replanning.
    """

    def __init__(
        self,
        llm,
        tools: list,
        risk_classifier: RuleBasedRiskClassifier,
        escalation_fn: EscalationCallback,
    ):
        """
        Args:
            llm: Language model to use.
            tools: Full tool set (read + write).
            risk_classifier: Classifies each tool call before execution.
            escalation_fn: Called when a high-risk action requires approval.
        """
        # Bind all tools so the LLM knows their schemas — the risk gate controls execution
        self.llm          = llm.bind_tools(tools)
        self.tool_map     = {t.name: t for t in tools}
        self.risk_fn      = risk_classifier
        self.escalation   = escalation_fn
        self.action_log   = []  # every tool call logged with its risk decision and outcome

    def run(self, task: str) -> str:
        """Execute a task with per-action risk classification.

        Args:
            task: The task description for the agent.

        Returns:
            The agent's final response string.
        """
        messages = [SystemMessage(content=SEMI_AUTO_SYSTEM), HumanMessage(content=task)]
        print(f'Task: {task}\n{"─" * 55}')

        for _ in range(20):  # iteration cap prevents runaway loops
            response = self.llm.invoke(messages)
            messages.append(response)

            # No tool calls means the agent has produced its final response
            if not response.tool_calls:
                print(f'\n{"─" * 55}\nFinal response:\n{response.content}')
                return response.content

            for tc in response.tool_calls:
                tool_name = tc['name']
                tool_args = tc['args']

                # ── RISK GATE: classify before every execution ──────────────
                risk = self.risk_fn.classify(tool_name, tool_args)

                # Start building the log entry for this tool call
                log_entry = {
                    'tool':               tool_name,
                    'args':               tool_args,
                    'risk_level':         risk.level.value,
                    'requires_approval':  risk.requires_approval,
                    'reason':             risk.reason,
                }

                if not risk.requires_approval:
                    # Low risk: execute immediately, no human involved
                    print(f'  [AUTO]     {tool_name} ({risk.level.value} risk)')
                    result = self.tool_map[tool_name].invoke(tool_args)
                    log_entry.update({'outcome': 'auto_executed', 'result': str(result)})

                else:
                    # High risk: pause and present to human for approval
                    approved, feedback = self.escalation(tool_name, tool_args, risk)

                    if approved:
                        print(f'  [APPROVED] {tool_name} — executing')
                        result = self.tool_map[tool_name].invoke(tool_args)
                        log_entry.update({'outcome': 'approved_executed', 'result': str(result)})
                    else:
                        print(f'  [REJECTED] {tool_name}. Feedback: {feedback}')
                        # Feed a rejection message back so the LLM can replan
                        result = (
                            f'Action "{tool_name}" was rejected by a human reviewer. '
                            f'Feedback: "{feedback}". Please propose an alternative approach.'
                        )
                        log_entry.update({'outcome': 'rejected', 'feedback': feedback})

                self.action_log.append(log_entry)
                messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))

        return 'Task did not complete within the iteration limit.'

The risk gate occupies exactly two lines in the loop: classify the tool call, then branch on `requires_approval`. Everything else - the LLM call, the tool execution, the message history - is the standard agentic pattern shared across all modes. The brevity of the gate is intentional: risk classification should not dominate the loop logic; it should be a transparent interceptor that most tool calls pass through without incident.

The `action_log` list accumulates one entry per tool call, regardless of outcome. Auto-executed calls, approved calls, and rejected calls all appear in the log with their classification reason and outcome. This is the raw data needed to answer the calibration question: are the thresholds set correctly?

## Demonstration: Q4 sales report pipeline
The task below mixes low-risk data retrieval with high-risk external communication. The agent will search the knowledge base, read sales data, and generate a report - all auto-executed - then attempt to send an email, which triggers escalation.

In [9]:
agent = SemiAutonomousAgent(
    llm=llm,
    tools=ALL_TOOLS,
    risk_classifier=rule_classifier,
    escalation_fn=auto_escalation,   # swap to console_escalation for interactive review
)

final_response = agent.run(
    'Prepare the Q4 2025 sales report and email it to vp-sales@company.com. '
    'Look up the contact preferences first, fetch the Q4 data, '
    'generate the report, then send it.'
)

Task: Prepare the Q4 2025 sales report and email it to vp-sales@company.com. Look up the contact preferences first, fetch the Q4 data, generate the report, then send it.
───────────────────────────────────────────────────────
  [AUTO]     search_knowledge_base (low risk)
  [AUTO]     read_sales_data (low risk)
  [AUTO]     generate_report (low risk)

ESCALATION REQUIRED
---------------------------------------------
Proposed action : send_email
Parameters      : {
  "to": "vp-sales@company.com",
  "subject": "Q4 2025 Sales Report",
  "body": "Dear VP of Sales,\n\nPlease find below the Q4 2025 Sales Report:\n\nREPORT: Q4 2025 Sales Report\nGenerated: 2026-03-27\n----------------------------------------\nIn Q4 2025, the sales performance was as follows:\n\n- Enterprise Customers: 1,200\n- Mid-Market Customers: 840\n- SMB Customers: 300\n\nTotal Annual Recurring Revenue (ARR) added: $4.2M.\n----------------------------------------\n\nBest regards,\n[Your Name]"
}
Risk level      : HIGH
Rea

The output shows the risk routing in action: the three read and compute tool calls print `[AUTO]` and proceed immediately; the `send_email` call prints the escalation block and then `AUTO-APPROVED` because `auto_escalation` was used. In an interactive session with `console_escalation`, the reviewer would see the exact email parameters - recipient, subject, body - and choose whether to approve.

## Inspecting the action log
Every tool call is recorded with its risk classification and outcome. This log serves two ongoing purposes: audit (every high-risk action has an explicit approval record) and calibration (the escalation rate tells you whether the thresholds are correctly set). A healthy escalation rate for most use cases is between 15% and 35% — high enough to catch genuine risks, low enough that reviewers engage rather than rubber-stamp.

In [10]:
print(f"{'#':<3} {'Tool':<30} {'Risk':<10} {'Outcome'}")
print('─' * 65)
for i, entry in enumerate(agent.action_log, 1):
    print(f"{i:<3} {entry['tool']:<30} {entry['risk_level']:<10} {entry['outcome']}")

# Summarise the routing statistics
total       = len(agent.action_log)
auto_count  = sum(1 for e in agent.action_log if e['outcome'] == 'auto_executed')
escal_count = sum(1 for e in agent.action_log if e['requires_approval'])
reject_count = sum(1 for e in agent.action_log if e['outcome'] == 'rejected')

print(f'\nSummary: {total} tool calls total')
print(f'  Auto-executed : {auto_count}  ({auto_count/total*100:.0f}% of calls)')
print(f'  Escalated     : {escal_count}  ({escal_count/total*100:.0f}% of calls)')
print(f'  Rejected      : {reject_count}')

# Calibration guidance
if total > 0:
    rate = escal_count / total
    if rate > 0.6:
        print('\nCalibration note: escalation rate > 60% — thresholds may be too conservative.')
    elif rate < 0.1:
        print('\nCalibration note: escalation rate < 10% — review false-negative rate.')
    else:
        print('\nCalibration note: escalation rate in healthy range (10–60%).')

#   Tool                           Risk       Outcome
─────────────────────────────────────────────────────────────────
1   search_knowledge_base          low        auto_executed
2   read_sales_data                low        auto_executed
3   generate_report                low        auto_executed
4   send_email                     high       approved_executed

Summary: 4 tool calls total
  Auto-executed : 3  (75% of calls)
  Escalated     : 1  (25% of calls)
  Rejected      : 0

Calibration note: escalation rate in healthy range (10–60%).


## When to use semi-autonomous mode

| Scenario | Why semi-autonomous mode fits |
|----------|-------------------------------|
| Task mixes read-heavy and write-heavy steps | Automate the safe majority; gate only the risky minority |
| Human review latency bottlenecks safe operations | Supervised mode phase checkpoints interrupt too much safe work |
| Agent has a proven track record on low-risk steps | Trust is established for the safe subset |
| Risk varies significantly across individual tool calls | Per-action routing is more precise than per-phase boundaries |

- **Two classifier strategies serve different needs.** Rule-based classification is O(1) and deterministic for tools whose risk is independent of arguments. LLM-driven classification handles tools where what the tool is called *with* changes the risk profile significantly.
- **Three outcomes per tool call.** Auto-execute (low risk), approved-execute (high risk + human approved), or rejected (high risk + human rejected, fed back as a replanning signal).
- **The action log enables calibration.** Escalation rate above 60% means thresholds are too conservative; below 10% means the classifier may be missing genuine risks. The log provides the data to tune continuously.
- **Transition path.** Promote from supervised mode when human approval rates approach 100% with no modifications. Promote to fully autonomous mode when escalation rates drop below roughly 5% with no quality incidents.